In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as opt
import matplotlib.pyplot as plt
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import wandb
import json

with open('secrets.json', 'r') as f:
    secrets = json.load(f)

wandb.login(key=secrets['WANDB_API_KEY'])

wb_key='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'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



PATH = './cifar_net.pth'

batch_size=64

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=2)

images, labels = next(iter(train_loader))
print(images.shape)  # torch.Size([64, 1, 28, 28])
print(labels.shape)  # torch.Size([64])


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/goose/.netrc
wandb: Currently logged in as: saisanka0621 (saisanka0621-ngc-lab) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
100.0%
100.0%
100.0%
100.0%


torch.Size([64, 1, 28, 28])
torch.Size([64])


In [2]:
class Net(nn.Module):
    def __init__(self,use_batchnorm=False):
        super(Net,self).__init__()
        self.conv1=nn.Conv2d(1,10,3)
        self.bn1=nn.BatchNorm2d(10) if use_batchnorm else nn.Identity()
        self.conv2=nn.Conv2d(10,20,3)
        self.bn2=nn.BatchNorm2d(20) if use_batchnorm else nn.Identity()

        self.fc1=nn.Linear(500,144)
        self.fc2=nn.Linear(144,72)
        self.fc3=nn.Linear(72,10)
    
    def forward(self,input):
        
        c1=F.relu(self.conv1(input))
        c1=self.bn1(c1)        
        s2=F.max_pool2d(c1,(2,2))
        c3=F.relu(self.conv2(s2))
        c3=self.bn2(c3)
        s4=F.max_pool2d(c3,2)
        s4=torch.flatten(s4,1)
        f5=F.relu(self.fc1(s4))
        f6=F.relu(self.fc2(f5))
        output=self.fc3(f6)
        return output



In [5]:


epoch=50
lr=1e-1

config={
    'epoch':epoch,
    'lr':lr
}

net=Net().to(device)
print(net)       

criterion=nn.CrossEntropyLoss()
optmizer=opt.Adam(net.parameters(),lr=lr)

def val_acc(net,test_loader):
    correct=0
    total=0
    net.eval()
    with torch.no_grad():
        for data in test_loader:
            images,labels=data
            images, labels = images.to(device), labels.to(device)
            outputs=net(images)
            _,predicted=torch.max(outputs,1)
            total+=labels.size(0)
            correct+=(predicted == labels).sum().item()
    net.train()
    return correct/total

wandb.init(project='lenet5exp',name=f'lr={lr}_Bn=F',config=config)
for epo in range(epoch):
    net.train()
    running_loss=0.0
    for i,data in enumerate(train_loader,0):
        inputs,labels=data
        inputs, labels = inputs.to(device), labels.to(device)
        optmizer.zero_grad()
        outputs=net(inputs)
        loss=criterion(outputs,labels)
        loss.backward()
        optmizer.step()

        running_loss+=loss.item()
        if i%20 == 19:
            print(f'[{epo+1},{i+1:5d}] loss={running_loss/20:.3f}')
    acc=val_acc(net,test_loader)
    wandb.log({"accuracy": acc, "loss": running_loss/len(train_loader)})
wandb.finish()
print('finish train')
torch.save(net.state_dict(), PATH)

Net(
  (conv1): Conv2d(1, 10, kernel_size=(3, 3), stride=(1, 1))
  (bn1): Identity()
  (conv2): Conv2d(10, 20, kernel_size=(3, 3), stride=(1, 1))
  (bn2): Identity()
  (fc1): Linear(in_features=500, out_features=144, bias=True)
  (fc2): Linear(in_features=144, out_features=72, bias=True)
  (fc3): Linear(in_features=72, out_features=10, bias=True)
)


[1,   20] loss=19.385
[1,   40] loss=21.697
[1,   60] loss=24.012
[1,   80] loss=26.318
[1,  100] loss=28.627
[1,  120] loss=30.933
[1,  140] loss=33.242
[1,  160] loss=35.553
[1,  180] loss=37.864
[1,  200] loss=40.177
[1,  220] loss=42.488
[1,  240] loss=44.800
[1,  260] loss=47.106
[1,  280] loss=49.415
[1,  300] loss=51.726
[1,  320] loss=54.031
[1,  340] loss=56.342
[1,  360] loss=58.660
[1,  380] loss=60.962
[1,  400] loss=63.266
[1,  420] loss=65.576
[1,  440] loss=67.886
[1,  460] loss=70.197
[1,  480] loss=72.506
[1,  500] loss=74.813
[1,  520] loss=77.123
[1,  540] loss=79.429
[1,  560] loss=81.738
[1,  580] loss=84.047
[1,  600] loss=86.354
[1,  620] loss=88.668
[1,  640] loss=90.978
[1,  660] loss=93.288
[1,  680] loss=95.601
[1,  700] loss=97.904
[1,  720] loss=100.220
[1,  740] loss=102.535
[1,  760] loss=104.841
[1,  780] loss=107.150
[1,  800] loss=109.457
[1,  820] loss=111.763
[1,  840] loss=114.072
[1,  860] loss=116.383
[1,  880] loss=118.695
[1,  900] loss=121.004


accuracy,▅█▄▄█▄█▃▅▄▃▅█▃▅▃▅▄█▄▅▃▅▄▄▄▄▅▃█▄▁▄▁▃█▄▄▃▄
loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
accuracy,0.1009
loss,2.30937


finish train


In [ ]:
net.load_state_dict(torch.load(PATH, weights_only=True))
def val_acc():
    correct=0
    total=0
    with torch.no_grad():
        for data in test_loader:
            images,labels=data
            outputs=net(images)
            _,predicted=torch.max(outputs,1)
            total+=labels.size(0)
            correct+=(predicted == labels).sum().item()
    return correct//total
